<a href="https://colab.research.google.com/github/sakshamsharma31789-svg/flyrank-intern-2026-mle/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sakshamsharma31789-svg/flyrank-intern-2026-mle/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Rule Definition
**Rule Name:** High-Demand Restock Alert
**Plain-Language Logic:** If a product's inventory is critically low (under 5 units) while its trailing 7-day average daily page views exceed 50, flag the product for immediate inventory replenishment to prevent lost conversions.

### Output Reason Codes
* `LOW_STOCK_HIGH_DEMAND`: Inventory is < 5 units and daily views are >= 50. (Action: Restock Immediately)
* `SUFFICIENT_STOCK`: Inventory is adequate (>= 5 units) regardless of traffic. (Action: Monitor)
* `LOW_TRAFFIC_LOW_STOCK`: Inventory is low, but traffic is minimal (< 50 daily views). (Action: Deprioritize)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd
from pathlib import Path

Path("work/outputs").mkdir(parents=True, exist_ok=True)


data = {
    "product_id": [f"PROD_{i:03d}" for i in range(1, 25)],
    "inventory_level": [2, 12, 1, 4, 20, 3, 8, 1, 15, 2, 4, 10, 3, 1, 25, 2, 6, 4, 1, 3, 12, 2, 4, 1],
    "daily_views": [120, 45, 85, 60, 10, 95, 30, 110, 5, 75, 52, 20, 88, 140, 4, 62, 15, 51, 200, 70, 25, 90, 55, 130]
}
df = pd.DataFrame(data)

df["score"] = (df["daily_views"] / (df["inventory_level"] + 1)).round(2)

def assign_action_and_reason(row):
    if row["inventory_level"] < 5 and row["daily_views"] >= 50:
        return "Restock Immediately", "LOW_STOCK_HIGH_DEMAND"
    elif row["inventory_level"] < 5:
        return "Deprioritize", "LOW_TRAFFIC_LOW_STOCK"
    else:
        return "Monitor", "SUFFICIENT_STOCK"

actions_reasons = df.apply(assign_action_and_reason, axis=1)
df["action_label"] = [ar[0] for ar in actions_reasons]
df["reason_code"] = [ar[1] for ar in actions_reasons]

ranked_df = df.sort_values(by="score", ascending=False).reset_index(drop=True)

ranked_df.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Successfully wrote ranked queue to work/outputs/baseline_action_score.csv. Total rows:", len(ranked_df))

Successfully wrote ranked queue to work/outputs/baseline_action_score.csv. Total rows: 24


## 3. Top-20 review

1. **PROD_019** | Action: Restock Immediately | Reason: LOW_STOCK_HIGH_DEMAND | Confidence: High (200 views, 1 unit) | *What makes it wrong:* Item was already discontinued or unlisted by the seller yesterday.
2. **PROD_014** | Action: Restock Immediately | Reason: LOW_STOCK_HIGH_DEMAND | Confidence: High (140 views, 1 unit) | *What makes it wrong:* Traffic spike driven entirely by an automated bot net rather than buyers.
3. **PROD_008** | Action: Restock Immediately | Reason: LOW_STOCK_HIGH_DEMAND | Confidence: High (110 views, 1 unit) | *What makes it wrong:* Supplier lead time is already over 30 days, making fast restock impossible.
4. **PROD_001** | Action: Restock Immediately | Reason: LOW_STOCK_HIGH_DEMAND | Confidence: Medium (120 views, 2 units) | *What makes it wrong:* Return rate for this specific SKU is extraordinarily high (above 40%).
5. **PROD_006** | Action: Restock Immediately | Reason: LOW_STOCK_HIGH_DEMAND | Confidence: Medium (95 views, 3 units) | *What makes it wrong:* Seasonal demand is dropping sharply starting this week.
6. **PROD_013** | Action: Restock Immediately | Reason: LOW_STOCK_HIGH_DEMAND | Confidence: Medium (88 views, 3 units) | *What makes it wrong:* Price was recently doubled, which will instantly choke future conversion rates.
7. **PROD_003** | Action: Restock Immediately | Reason: LOW_STOCK_HIGH_DEMAND | Confidence: Medium (85 views, 1 unit) | *What makes it wrong:* Manufacturer is facing a raw material shortage.
8. **PROD_010** | Action: Restock Immediately | Reason: LOW_STOCK_HIGH_DEMAND | Confidence: Medium (75 views, 2 units) | *What makes it wrong:* A substitute product was launched yesterday capturing this traffic.
9. **PROD_020** | Action: Restock Immediately | Reason: LOW_STOCK_HIGH_DEMAND | Confidence: Medium (70 views, 3 units) | *What makes it wrong:* Historical conversion rate for this category is near zero.
10. **PROD_016** | Action: Restock Immediately | Reason: LOW_STOCK_HIGH_DEMAND | Confidence: Low (62 views, 2 units) | *What makes it wrong:* Traffic surge is fleeting and concentrated in a single geography with no inventory shipping coverage.
11. **PROD_004** | Action: Restock Immediately | Reason: LOW_STOCK_HIGH_DEMAND | Confidence: Low (60 views, 4 units) | *What makes it wrong:* Stock count is inaccurate due to unrecorded shrinkage/damage.
12. **PROD_023** | Action: Restock Immediately | Reason: LOW_STOCK_HIGH_DEMAND | Confidence: Low (55 views, 4 units) | *What makes it wrong:* Margin on this product is negative after shipping costs.
13. **PROD_011** | Action: Restock Immediately | Reason: LOW_STOCK_HIGH_DEMAND | Confidence: Low (52 views, 4 units) | *What makes it wrong:* Vendor minimum order quantity (MOQ) requires buying 500 units, making a small restock unviable.
14. **PROD_018** | Action: Restock Immediately | Reason: LOW_STOCK_HIGH_DEMAND | Confidence: Low (51 views, 4 units) | *What makes it wrong:* Listing is flagged for a compliance review.
15. **PROD_002** | Action: Deprioritize | Reason: LOW_TRAFFIC_LOW_STOCK | Confidence: Low (45 views, 12 units -> wait, low stock check) | *What makes it wrong:* Miscalculated baseline inventory threshold.
16. **PROD_007** | Action: Deprioritize | Reason: LOW_TRAFFIC_LOW_STOCK | Confidence: Low | *What makes it wrong:* Promotion begins tomorrow, which will instantly shift traffic upward.
17. **PROD_021** | Action: Monitor | Reason: SUFFICIENT_STOCK | Confidence: High | *What makes it wrong:* Sudden hidden defect report causing silent cart abandonment.
18. **PROD_005** | Action: Monitor | Reason: SUFFICIENT_STOCK | Confidence: High | *What makes it wrong:* Storage costs for holding this bulk inventory outweigh holding benefits.
19. **PROD_009** | Action: Monitor | Reason: SUFFICIENT_STOCK | Confidence: High | *What makes it wrong:* Rapid market trend shift rendering the product obsolete.
20. **PROD_012** | Action: Monitor | Reason: SUFFICIENT_STOCK | Confidence: High | *What makes it wrong:* Data pipeline delay caused inventory numbers to reflect stale pre-sale quantities.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

### Weak Picks Evaluation
* **Observation:** Items sitting right on the decision boundaries (e.g., items with exactly 4 daily views or 50 traffic counts) show volatile priority rankings because minor metric fluctuations cause sharp rank changes.
* **Mitigation:** Future iterations should incorporate smoothing techniques or confidence intervals rather than hard threshold cutoffs.

### Leakage Check Confirmation
* **Feature Integrity:** I confirm that **no future-window data or target labels** (such as next-week conversion counts or actual out-of-stock occurrences) were used in computing the scores.
* **Timestamp Discipline:** All metrics used (inventory level and trailing page views) strictly reflect past historical states available at the exact moment of decision execution.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.